In [19]:
from google.colab import files
import pandas as pd

train_base = pd.read_csv(" ", low_memory=False)
train_base = pd.read_csv(" ", low_memory=False)
test_base = pd.read_csv("/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_base.csv", low_memory=False)


# Load both pieces or files of the static_0 training table
static_train_0 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/train/train_static_0_0.csv",
    low_memory=False
)

static_train_1 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/train/train_static_0_1.csv",
    low_memory=False
)

# Stack or combine both pieces or files into one complete training static table
static_train = pd.concat(
    [static_train_0, static_train_1],
    ignore_index=True
)

# Load both pieces of the static_0 test table
static_test_0 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_static_0_0.csv",
    low_memory=False
)

static_test_1 = pd.read_csv(
    "/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_static_0_1.csv",
    low_memory=False
)

# Stack both pieces into one complete test static table
static_test = pd.concat(
    [static_test_0, static_test_1],
    ignore_index=True
)
train = train_base.merge(
    static_train,
    on="case_id",
    how="left",
    validate="one_to_one"
)

test = test_base.merge(
    static_test,
    on="case_id",
    how="left",
    validate="one_to_one"
)


print(train.shape)
print(test.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/train/train_static_0_0.csv'

In [ ]:
features = [
    "mainoccupationinc_384A",
    "credamount_770A",
    "annuity_780A",
    "days_employed_700P",
    "education_927M",
    "maritalstatus_703M",
]

print("Missingness:")
print((train[features].isna().mean() * 100).round(1))

print("\nNumeric summary:")
print(
    train[
        [
            "mainoccupationinc_384A",
            "credamount_770A",
            "annuity_780A",
            "days_employed_700P",
        ]
    ].describe(percentiles=[0.01, 0.50, 0.99])
)

## EDA findings and cleaning rules

### Missing values would include the following:

- `mainoccupationinc_384A`: 5.0% missing
- `credamount_770A`: 2.0% missing
- `annuity_780A`: 10.0% missing
- `days_employed_700P`: 15.1% missing
- `education_927M` and `maritalstatus_703M`: 0% missing

### Numeric values and outliers

The numeric amount fields are right-skewed: their maximum values are much larger than their 99th-percentile values. For example, income has a 99th percentile of about 64,418 but a maximum of 174,473.

No rows will be removed or eliminated for this initial baseline. Numeric missing values will be median-imputed and numeric variables will use `RobustScaler`, which is less sensitive or responsive to extreme values.

In [29]:
from google.colab import files

import pandas as pd


class SimplePreprocessor:

    def load_csv(self, file_path):
        # Read any CSV file and transform into a pandas DataFrame
        return pd.read_csv(file_path)

    def fit(self, df):
        # Identify numeric and categorical columns
        self.numeric_columns = df.select_dtypes(include="number").columns.tolist()
        self.categorical_columns = df.select_dtypes(
            exclude="number"
        ).columns.tolist()

        # Save median values for numeric missing values
        self.numeric_medians = df[self.numeric_columns].median()

        # Save most common value for categorical missing values
        self.categorical_modes = {}

        for column in self.categorical_columns:
            if df[column].dropna().empty:
                self.categorical_modes[column] = "Missing"
            else:
                self.categorical_modes[column] = df[column].mode().iloc[0]

        # Save IQR outlier limits for numeric columns
        self.outlier_limits = {}

        for column in self.numeric_columns:
            q1 = df[column].quantile(0.25)
            q3 = df[column].quantile(0.75)
            iqr = q3 - q1

            lower_limit = q1 - 1.5 * iqr
            upper_limit = q3 + 1.5 * iqr

            self.outlier_limits[column] = (lower_limit, upper_limit)

        return self

    def transform(self, df, cap_outliers=False):
        # Work on a copy so the original DataFrame is unchanged
        cleaned_df = df.copy()

        # Fill numeric missing values with saved medians
        for column in self.numeric_columns:
            cleaned_df[column] = cleaned_df[column].fillna(
                self.numeric_medians[column]
            )

        # Fill categorical missing values with saved modes
        for column in self.categorical_columns:
            cleaned_df[column] = cleaned_df[column].fillna(
                self.categorical_modes[column]
            )

        # Optional: cap outliers using saved IQR limits
        if cap_outliers:
            for column in self.numeric_columns:
                lower_limit, upper_limit = self.outlier_limits[column]

                cleaned_df[column] = cleaned_df[column].clip(
                    lower=lower_limit,
                    upper=upper_limit
                )

        return cleaned_df

    def fit_transform(self, df, cap_outliers=False):
        self.fit(df)
        return self.transform(df, cap_outliers=cap_outliers)

    def show_eda(self, df):
        # Print quick EDA information
        print("Shape:", df.shape)

        print("\nColumn data types:")
        print(df.dtypes)

        print("\nMissing values:")
        print(df.isna().sum())

        print("\nNumeric summary:")
        print(df.select_dtypes(include="number").describe())


    # Read a CSV file
df = pd.read_csv("/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_base.csv")

# Create and run the preprocessing pipeline
preprocessor = SimplePreprocessor()

clean_df = preprocessor.fit_transform(
    df,
    cap_outliers=False
)

# Check that cleaning worked
print("Missing values after cleaning:", clean_df.isna().sum().sum())

# Save the cleaned CSV file
clean_df.to_csv("cleaned_file.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/skylarroberts/Downloads/synth_home_credit (1)/csv_files/test/test_base.csv'